# 02 - Preprocessing

Clean the dataset, handle outliers, encode categoricals, split, and scale.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os
import sys

# Add src to path
sys.path.append(os.path.abspath('..'))
from src.preprocess import handle_outliers, encode_categorical


### 1. Load Data


In [ ]:
df = pd.read_csv('../data/heart.csv')
print("Initial shape:", df.shape)


### 2. Missing Values


In [ ]:
print(df.isnull().sum())
# No missing values in this dataset natively, but if there were, we'd handle them.


### 3. Duplicate Rows


In [ ]:
duplicates = df.duplicated().sum()
print(f"Duplicates found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()


### 4. Handle Outliers (Cholesterol = 0)


In [ ]:
print("Zeros in Cholesterol before:", (df['Cholesterol'] == 0).sum())
df = handle_outliers(df)
print("Zeros in Cholesterol after:", (df['Cholesterol'] == 0).sum())


### 5. Encode Categorical Columns


In [ ]:
categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
df_encoded = encode_categorical(df, columns=categorical_cols)
print("Shape after encoding:", df_encoded.shape)
display(df_encoded.head())


### 6. Separate Features and Target


In [ ]:
X = df_encoded.drop(columns=['HeartDisease'])
y = df_encoded['HeartDisease']


### 7 & 8. Split into Train/Test and Scale


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
scaler = StandardScaler()

# Fit only on train to prevent data leakage
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])


### 9 & 10. Save Preprocessed Data and Print Shapes


In [ ]:
print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/processed_data.pkl', 'wb') as f:
    pickle.dump((X_train_scaled, X_test_scaled, y_train, y_test, scaler), f)

print("Preprocessed data saved to data/processed/processed_data.pkl")
